# NLP processing

In [1]:
import numpy as np
import pandas as pd
import nltk
import jieba
from sudachipy import tokenizer
from sudachipy import dictionary
import re

In [2]:
# import data

jp = pd.read_csv('data/jp.csv')

en = pd.read_csv('data/en.csv')

cn = pd.read_csv('data/cn.csv')



In [3]:
# read NRC emotion lexicon - available in multiple languages through google translate 

nrc = pd.read_csv('data/NRC-Emotion-Lexicon-v0.92-InManyLanguages-web.csv')


In [4]:
nrc = nrc[['English Word', 
                   'Chinese (simplified) Translation (Google Translate)',
                   'Japanese Translation (Google Translate)',
                   'Positive', 
                   'Negative', 
                   'Anger',
                   'Anticipation', 
                   'Disgust', 
                   'Fear', 
                   'Joy', 
                   'Sadness', 
                   'Surprise',
                   'Trust'
                   ]]
                

# Clean & Tokenize - ENG

In [5]:
en = en[['label', 'merged_text']]

# removes url, punctuations, converts to lower-case. 
def clean_for_emotion(text):
    import re
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    return text

en['cleaned_text'] = en['merged_text'].apply(clean_for_emotion)

# remove punctuations while preserving some, then tokenize 
from nltk.tokenize import word_tokenize

en['tokenized'] = en['cleaned_text'].apply(lambda x: re.findall(r"\b\w+(?:'\w+)?\b", x))

en.head()

,label,merged_text,cleaned_text,tokenized
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,law enforcement on high alert following threat...,"[law, enforcement, on, high, alert, following,..."
1,0,TITLE: BODY: Did they post their votes for H...,title: body: did they post their votes for h...,"[title, body, did, they, post, their, votes, f..."
2,0,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,unbelievable! obama’s attorney general says mo...,"[unbelievable, obama, s, attorney, general, sa..."
3,1,"Bobby Jindal, raised Hindu, uses story of Chri...","bobby jindal, raised hindu, uses story of chri...","[bobby, jindal, raised, hindu, uses, story, of..."
4,0,SATAN 2: Russia unvelis an image of its terrif...,satan 2: russia unvelis an image of its terrif...,"[satan, 2, russia, unvelis, an, image, of, its..."


# Clean & Tokenize - CN

In [6]:
cn = cn[['label', 'merged_text']]

# Clean URLs and remove punctuation
def clean_text_cn(text):
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove punctuation (both English and Chinese)
    text = re.sub(r"[^\w\s]", "", text)
    return text

cn['cleaned_text'] = cn['merged_text'].apply(clean_text_cn)

# Tokenize with jieba
cn['tokenized'] = cn['cleaned_text'].apply(jieba.lcut)

cn.head()


Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/hn/0nlgxbrs6cq1l75k7sqk3kf40000gn/T/jieba.cache
Loading model cost 0.511 seconds.
Prefix dict has been built successfully.


,label,merged_text,cleaned_text,tokenized
0,1,离婚窗口排长队,离婚窗口排长队,"[离婚, 窗口, 排长队, ]"
1,1,哈工程教学楼地下有数门“超电磁炮”,哈工程教学楼地下有数门超电磁炮,"[哈, 工程, 教学楼, 地下, 有数, 门超, 电磁炮, ]"
2,1,薯片胀袋是变质了,薯片胀袋是变质了,"[薯片, 胀, 袋, 是, 变质, 了, ]"
3,1,成本只要1毛钱的人造鸡蛋泛滥,成本只要1毛钱的人造鸡蛋泛滥,"[成本, 只要, 1, 毛钱, 的, 人造, 鸡蛋, 泛滥, ]"
4,1,隔着玻璃晒太阳能补钙,隔着玻璃晒太阳能补钙,"[隔, 着, 玻璃, 晒太阳, 能, 补钙, ]"


# Clean & Tokenize - JP

In [7]:
# An Experimental Evaluation of Japanese Tokenizers for Sentiment-Based Text Classification - Sudachi as a tokenizer that works well

jp = jp[['label', 'merged_text']]

jp['cleaned_text'] = jp['merged_text'].apply(clean_text_cn)

tokenizer_obj = dictionary.Dictionary().create()

jp.head()

,label,merged_text,cleaned_text
0,0,朝日新聞など各社の報道によれば、宅配便最大手「ヤマト運輸」が日本郵政公社を相手取り、大手コン...,朝日新聞など各社の報道によれば宅配便最大手ヤマト運輸が日本郵政公社を相手取り大手コンビニエン...
1,1,11月5日の各社報道によると、諫早湾干拓事業は諫早海人（諫早湾の「海」）に囲まれる大洋に位置...,11月5日の各社報道によると諫早湾干拓事業は諫早海人諫早湾の海に囲まれる大洋に位置することか...
2,1,産経新聞、中日新聞によると、2004年から2005年まで、この大会による3年おきの開催を、2...,産経新聞中日新聞によると2004年から2005年までこの大会による3年おきの開催を2006年...
3,1,開催地のリオデジャネイロ市に対して、大会期間中のリオデジャネイロオリンピックに関する公式発表...,開催地のリオデジャネイロ市に対して大会期間中のリオデジャネイロオリンピックに関する公式発表は...
4,1,毎日新聞・時事通信によると、2006年2月13日には、グッドウィル・グッゲンハイム・アン・ハ...,毎日新聞時事通信によると2006年2月13日にはグッドウィルグッゲンハイムアンハルクを経営す...


In [8]:
mode = tokenizer.Tokenizer.SplitMode.C

jp['tokenized'] = jp['cleaned_text'].apply(lambda text: [m.surface() for m in tokenizer_obj.tokenize(text, mode)])

jp.head()

,label,merged_text,cleaned_text,tokenized
0,0,朝日新聞など各社の報道によれば、宅配便最大手「ヤマト運輸」が日本郵政公社を相手取り、大手コン...,朝日新聞など各社の報道によれば宅配便最大手ヤマト運輸が日本郵政公社を相手取り大手コンビニエン...,"[朝日新聞, など, 各社, の, 報道, に, よれ, ば, 宅配便, 最大手, ヤマト運..."
1,1,11月5日の各社報道によると、諫早湾干拓事業は諫早海人（諫早湾の「海」）に囲まれる大洋に位置...,11月5日の各社報道によると諫早湾干拓事業は諫早海人諫早湾の海に囲まれる大洋に位置することか...,"[11, 月, 5, 日, の, 各社, 報道, に, よる, と, 諫早湾, 干拓, 事業..."
2,1,産経新聞、中日新聞によると、2004年から2005年まで、この大会による3年おきの開催を、2...,産経新聞中日新聞によると2004年から2005年までこの大会による3年おきの開催を2006年...,"[産経新聞, 中日, 新聞, に, よる, と, 2004, 年, から, 2005, 年,..."
3,1,開催地のリオデジャネイロ市に対して、大会期間中のリオデジャネイロオリンピックに関する公式発表...,開催地のリオデジャネイロ市に対して大会期間中のリオデジャネイロオリンピックに関する公式発表は...,"[開催地, の, リオデジャネイロ, 市, に, 対し, て, 大会, 期間中, の, リオ..."
4,1,毎日新聞・時事通信によると、2006年2月13日には、グッドウィル・グッゲンハイム・アン・ハ...,毎日新聞時事通信によると2006年2月13日にはグッドウィルグッゲンハイムアンハルクを経営す...,"[毎日, 新聞, 時事, 通信, に, よる, と, 2006, 年, 2, 月, 13, ..."


In [9]:
type(jp['tokenized'].iloc[0])

list

# Emotion Detection 

looks up each token in the NRC dict, calculate a normalized emotion score for each news input.

In [10]:
nrc = nrc.rename(columns = {"English Word":'EN',
                      'Chinese (simplified) Translation (Google Translate)':'CN',
                      'Japanese Translation (Google Translate)':'JP'})
nrc.head()

,EN,CN,JP,Positive,Negative,Anger,Anticipation,Disgust,Fear,Joy,Sadness,Surprise,Trust
0,aback,吓了一跳,あっけ,0,0,0,0,0,0,0,0,0,0
1,abacus,算盘,そろばん,0,0,0,0,0,0,0,0,0,1
2,abandon,放弃,捨てます,0,1,0,0,0,1,0,1,0,0
3,abandoned,弃,放棄されました,0,1,1,0,0,1,0,1,0,0
4,abandonment,放弃,放棄,0,1,1,0,0,1,0,1,1,0


In [16]:
EMOTIONS = ['Positive', 
            'Negative',
            'Anger',
            'Anticipation',
            'Disgust',
            'Fear',
            'Joy',
            'Sadness',
            'Surprise',
            'Trust']


LANGUAGES = {
    'English': (en, 'EN'),           
    'Chinese': (cn, 'CN'), 
    'Japanese': (jp, 'JP'),          
}

# build lookup table
def build_lookup(lex_df, word_col):
    """Build {word: {emotion: score}} dict for fast lookup."""
    sub = lex_df.dropna(subset=[word_col]).drop_duplicates(subset=[word_col], keep='first').set_index(word_col)[EMOTIONS]
    return sub.to_dict('index')


In [17]:
# scoring tokens
def score_tokens(token_list, lookup):
    """
    Given a list of tokens and a lookup dict,
    return a dict of normalised emotion scores (per token).
    Normalising by article length reduces bias from longer articles.
    """
    counts = {e: 0 for e in EMOTIONS}
    for token in token_list:
        if token in lookup:
            for e in EMOTIONS:
                counts[e] += lookup[token][e]

    n = len(token_list)
    return {e: counts[e]/n for e in EMOTIONS}

# add emotions score

def add_emo_score(df, lang_col):
    """Parse tokenized_text and add one column per emotion."""
    lookup = build_lookup(nrc, lang_col)
    df = df.copy()
    scores = df['tokenized'].apply(lambda x: score_tokens(x, lookup))
    scores_df = pd.DataFrame(scores.tolist(), index = df.index)
    df = pd.concat([df.drop(columns = ['tokenized']), scores_df], axis = 1)
    return df 



In [18]:
# apply to three languages

scored = {}

for lang, (df, col) in LANGUAGES.items():
    scored[lang] = add_emo_score(df, col)

scored

{'English':        label                                        merged_text  \
 0          0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
 1          0   TITLE:  BODY: Did they post their votes for H...   
 2          0  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
 3          1  Bobby Jindal, raised Hindu, uses story of Chri...   
 4          0  SATAN 2: Russia unvelis an image of its terrif...   
 ...      ...                                                ...   
 72129      1  Russians steal research on Trump in hack of U....   
 72130      0   WATCH: Giuliani Demands That Democrats Apolog...   
 72131      1  Migrants Refuse To Leave Train At Refugee Camp...   
 72132      1  Trump tussle gives unpopular Mexican leader mu...   
 72133      0  Goldman Sachs Endorses Hillary Clinton For Pre...   
 
                                             cleaned_text  Positive  Negative  \
 0      law enforcement on high alert following threat...  0.050420  0.061975   
 1       

In [21]:
# aggregate tables 

agg_tables = {}
for lang, df in scored.items():
    score_tables = df.groupby('label')[EMOTIONS].mean().T
    score_tables.columns = ['real (0)', 'fake (1)']
    score_tables['diff(fake-real)'] = score_tables['fake (1)'] - score_tables['real (0)']
    agg_tables[lang] = score_tables
    print(f"\n── {lang} ──")
    print(score_tables.round(5))


── English ──
              real (0)  fake (1)  diff(fake-real)
Positive       0.05839   0.06181          0.00341
Negative       0.04035   0.03635         -0.00401
Anger          0.02094   0.01829         -0.00264
Anticipation   0.02372   0.02370         -0.00002
Disgust        0.01216   0.00825         -0.00391
Fear           0.02676   0.02534         -0.00142
Joy            0.01583   0.01469         -0.00114
Sadness        0.01739   0.01542         -0.00196
Surprise       0.01827   0.01568         -0.00259
Trust          0.04427   0.05047          0.00620

── Chinese ──
              real (0)  fake (1)  diff(fake-real)
Positive       0.05292   0.04764         -0.00528
Negative       0.03611   0.04962          0.01351
Anger          0.01560   0.02101          0.00541
Anticipation   0.02283   0.01904         -0.00379
Disgust        0.01076   0.01760          0.00684
Fear           0.02110   0.02885          0.00775
Joy            0.01607   0.01280         -0.00328
Sadness        0.015